# 03 - Training and Evaluation

This notebook trains candidate models, compares validation performance, and saves production artifacts.


In [1]:
# Cell 0: Setup
from __future__ import annotations

import importlib.util
import json
from datetime import UTC, datetime
from pathlib import Path
import sys

required_libs = ['numpy', 'pandas', 'sklearn', 'joblib', 'scipy']
missing_libs = [lib for lib in required_libs if importlib.util.find_spec(lib) is None]
if missing_libs:
    raise RuntimeError(
        'Missing required packages for Notebook 03: '
        + ', '.join(missing_libs)
        + '\nInstall with:\napps/backend/.venv/bin/python -m pip install -r ml/requirements.txt'
    )

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / 'ml' / 'pipelines' / 'churn_train.py').exists():
        ROOT = p
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ml.pipelines.churn_train import split_by_snapshot

FEATURES_PATH = ROOT / 'ml' / 'data' / 'churn_training_features.csv'
if not FEATURES_PATH.exists():
    FEATURES_PATH = ROOT / 'ml' / 'data' / 'churn_training_dataset.csv'

SELECTED_PATH = ROOT / 'ml' / 'models' / 'selected_features_notebook.json'
RANKING_PATH = ROOT / 'ml' / 'models' / 'feature_ranking_notebook.csv'

assert FEATURES_PATH.exists(), 'Run notebook 01/02 first.'
assert SELECTED_PATH.exists(), 'Run notebook 02 first.'
assert RANKING_PATH.exists(), 'Run notebook 02 first.'

df = pd.read_csv(FEATURES_PATH)
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'], utc=True)
df['will_order_next_30d'] = df['will_order_next_30d'].astype(int)
df = df.sort_values(['snapshot_date', 'user_id']).reset_index(drop=True)

selected_payload = json.loads(SELECTED_PATH.read_text(encoding='utf-8'))
selected_features = selected_payload['selected_features']
ranking_df = pd.read_csv(RANKING_PATH)

missing_selected = [c for c in selected_features if c not in df.columns]
if missing_selected:
    raise RuntimeError(
        'Selected features missing from dataset: '
        + ', '.join(missing_selected)
        + '\nRe-run notebook 02 to regenerate features and selection outputs.'
    )

print('Loaded dataset:', df.shape)
print('Snapshot count:', df['snapshot_date'].nunique())
print('Target rate:', round(df['will_order_next_30d'].mean(), 4))
print('Notebook 02 selected features:', len(selected_features))




Loaded dataset: (862, 93)
Snapshot count: 12
Target rate: 0.5162
Notebook 02 selected features: 24


In [2]:
# Cell 1: Time split + selected-feature anti-duplication pruning
train_df, val_df, test_df = split_by_snapshot(df)

y_train = train_df['will_order_next_30d']
y_val = val_df['will_order_next_30d']
y_test = test_df['will_order_next_30d']

selected_numeric = [
    c for c in selected_features if pd.api.types.is_numeric_dtype(df[c])
]
dropped_non_numeric = sorted(set(selected_features) - set(selected_numeric))

score_col = 'final_score' if 'final_score' in ranking_df.columns else 'perm_importance'
score_map = ranking_df.set_index('feature')[score_col].to_dict()
default_score = (min(score_map.values()) - 1.0) if score_map else -1.0

corr = train_df[selected_numeric].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

corr_drop = set()
high_corr_pairs = []
for b in upper.columns:
    for a, abs_corr in upper[b].dropna().items():
        if abs_corr <= 0.995:
            continue
        score_a = score_map.get(a, default_score)
        score_b = score_map.get(b, default_score)
        drop = a if score_a < score_b else b
        keep = b if drop == a else a
        corr_drop.add(drop)
        high_corr_pairs.append({'keep': keep, 'drop': drop, 'abs_corr': float(abs_corr)})

selected_model = [c for c in selected_numeric if c not in corr_drop]
if len(selected_model) < 8:
    selected_model = selected_numeric

X_train = train_df[selected_model]
X_val = val_df[selected_model]
X_test = test_df[selected_model]

split_profile = pd.DataFrame(
    [
        {
            'split': 'train',
            'rows': len(train_df),
            'snapshots': train_df['snapshot_date'].nunique(),
            'start': train_df['snapshot_date'].min(),
            'end': train_df['snapshot_date'].max(),
            'target_rate': float(y_train.mean()),
        },
        {
            'split': 'val',
            'rows': len(val_df),
            'snapshots': val_df['snapshot_date'].nunique(),
            'start': val_df['snapshot_date'].min(),
            'end': val_df['snapshot_date'].max(),
            'target_rate': float(y_val.mean()),
        },
        {
            'split': 'test',
            'rows': len(test_df),
            'snapshots': test_df['snapshot_date'].nunique(),
            'start': test_df['snapshot_date'].min(),
            'end': test_df['snapshot_date'].max(),
            'target_rate': float(y_test.mean()),
        },
    ]
)

display(split_profile)
print('Selected features (input):', len(selected_features))
print('Selected numeric:', len(selected_numeric))
print('Dropped non-numeric:', dropped_non_numeric)
print('Dropped near-duplicate selected features:', len(corr_drop))
print('Final model feature count:', len(selected_model))
if high_corr_pairs:
    display(pd.DataFrame(high_corr_pairs).sort_values('abs_corr', ascending=False).head(20))



,split,rows,snapshots,start,end,target_rate
0,train,355,7,2025-11-20 00:00:00+00:00,2026-01-01 00:00:00+00:00,0.591549
1,val,162,2,2026-01-08 00:00:00+00:00,2026-01-15 00:00:00+00:00,0.530864
2,test,345,3,2026-01-22 00:00:00+00:00,2026-02-05 00:00:00+00:00,0.431884


Selected features (input): 24
Selected numeric: 24
Dropped non-numeric: []
Dropped near-duplicate selected features: 0
Final model feature count: 24


In [3]:
# Cell 2: Train candidate models

def safe_roc_auc(y_true, y_score):
    return float(roc_auc_score(y_true, y_score)) if y_true.nunique() > 1 else 0.5


candidates = {
    'logistic_regression': Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter=5000, class_weight='balanced', solver='lbfgs')),
        ]
    ),
    'random_forest': Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            (
                'clf',
                RandomForestClassifier(
                    n_estimators=700,
                    max_depth=12,
                    min_samples_leaf=6,
                    random_state=42,
                    n_jobs=-1,
                    class_weight='balanced_subsample',
                ),
            ),
        ]
    ),
    'extra_trees': Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            (
                'clf',
                ExtraTreesClassifier(
                    n_estimators=900,
                    max_depth=14,
                    min_samples_leaf=5,
                    random_state=42,
                    n_jobs=-1,
                    class_weight='balanced_subsample',
                ),
            ),
        ]
    ),
}

rows = []
trained = {}
for name, model in candidates.items():
    model.fit(X_train, y_train)
    val_probs = model.predict_proba(X_val)[:, 1]
    rows.append(
        {
            'model': name,
            'val_roc_auc': safe_roc_auc(y_val, val_probs),
            'val_pr_auc': float(average_precision_score(y_val, val_probs)),
            'val_brier': float(brier_score_loss(y_val, val_probs)),
        }
    )
    trained[name] = model

val_table = pd.DataFrame(rows).sort_values(
    ['val_pr_auc', 'val_roc_auc', 'val_brier'],
    ascending=[False, False, True],
).reset_index(drop=True)
display(val_table)



,model,val_roc_auc,val_pr_auc,val_brier
0,extra_trees,0.993268,0.992753,0.030041
1,random_forest,0.992350,0.991549,0.031455
2,logistic_regression,0.980416,0.975087,0.046647


In [4]:
# Cell 3: Threshold analysis (optimize F2 with precision floor)
best_name = val_table.iloc[0]['model']
best_model = trained[best_name]

val_probs = best_model.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, val_probs)

if len(thresholds) == 0:
    threshold = 0.5
    threshold_policy = 'fallback_0.5'
    threshold_table = pd.DataFrame(
        [
            {
                'threshold': threshold,
                'precision': float(precision[-1]),
                'recall': float(recall[-1]),
                'f1': 0.0,
                'f2': 0.0,
                'predicted_positive_rate': float((val_probs >= threshold).mean()),
            }
        ]
    )
else:
    threshold_table = pd.DataFrame(
        {
            'threshold': thresholds,
            'precision': precision[:-1],
            'recall': recall[:-1],
        }
    )
    threshold_table['f1'] = 2 * threshold_table['precision'] * threshold_table['recall'] / (
        threshold_table['precision'] + threshold_table['recall'] + 1e-12
    )
    threshold_table['f2'] = (
        (1 + 2**2)
        * threshold_table['precision']
        * threshold_table['recall']
        / (4 * threshold_table['precision'] + threshold_table['recall'] + 1e-12)
    )
    threshold_table['predicted_positive_rate'] = threshold_table['threshold'].apply(
        lambda t: float((val_probs >= t).mean())
    )

    min_precision = 0.40
    eligible = threshold_table[threshold_table['precision'] >= min_precision]
    if not eligible.empty:
        chosen = eligible.sort_values(['f2', 'f1', 'recall'], ascending=False).iloc[0]
        threshold_policy = f'max_f2_with_precision_floor_{min_precision:.2f}'
    else:
        chosen = threshold_table.sort_values(['f1', 'recall'], ascending=False).iloc[0]
        threshold_policy = 'max_f1_no_precision_floor'

    threshold = float(chosen['threshold'])

print('Best model:', best_name)
print('Threshold policy:', threshold_policy)
print('Selected threshold:', round(threshold, 6))

display(threshold_table.sort_values(['f2', 'f1'], ascending=False).head(15))



Best model: extra_trees
Threshold policy: max_f2_with_precision_floor_0.40
Selected threshold: 0.312471


,threshold,precision,recall,f1,f2,predicted_positive_rate
75,0.312471,0.988372,0.988372,0.988372,0.988372,0.530864
74,0.295807,0.977011,0.988372,0.982659,0.986079,0.537037
73,0.286840,0.965909,0.988372,0.977011,0.983796,0.543210
67,0.200031,0.914894,1.000000,0.955556,0.981735,0.580247
72,0.265723,0.955056,0.988372,0.971429,0.981524,0.549383
66,0.197037,0.905263,1.000000,0.950276,0.979499,0.586420
71,0.264350,0.944444,0.988372,0.965909,0.979263,0.555556
76,0.554293,0.988235,0.976744,0.982456,0.979021,0.524691
65,0.174489,0.895833,1.000000,0.945055,0.977273,0.592593
70,0.253194,0.934066,0.988372,0.960452,0.977011,0.561728


In [5]:
# Cell 4: Validation and test evaluation + snapshot diagnostics

def eval_metrics(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)
    return {
        'roc_auc': float(roc_auc_score(y_true, probs)) if y_true.nunique() > 1 else 0.5,
        'pr_auc': float(average_precision_score(y_true, probs)),
        'brier': float(brier_score_loss(y_true, probs)),
        'accuracy': float(accuracy_score(y_true, pred)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'f1': float(f1_score(y_true, pred, zero_division=0)),
        'f2': float(fbeta_score(y_true, pred, beta=2, zero_division=0)),
        'predicted_positive_rate': float(pred.mean()),
    }

val_metrics = eval_metrics(y_val, val_probs, threshold)
test_probs = best_model.predict_proba(X_test)[:, 1]
test_metrics = eval_metrics(y_test, test_probs, threshold)

snapshot_rows = []
for split_name, frame, probs in [
    ('val', val_df, val_probs),
    ('test', test_df, test_probs),
]:
    temp = pd.DataFrame(
        {
            'snapshot_date': frame['snapshot_date'].values,
            'target': frame['will_order_next_30d'].values,
            'score': probs,
        }
    )
    grouped = (
        temp.groupby('snapshot_date')
        .agg(
            users=('target', 'size'),
            target_rate=('target', 'mean'),
            avg_score=('score', 'mean'),
            score_std=('score', 'std'),
        )
        .reset_index()
        .sort_values('snapshot_date')
    )
    grouped['predicted_positive_rate'] = temp.groupby('snapshot_date')['score'].apply(
        lambda s: float((s >= threshold).mean())
    ).values
    grouped['split'] = split_name
    grouped['target_rate_index'] = grouped['target_rate'] / max(grouped['target_rate'].iloc[0], 1e-12)
    snapshot_rows.append(grouped)

snapshot_metrics = pd.concat(snapshot_rows, ignore_index=True)

print('Validation metrics:', val_metrics)
print('Test metrics:', test_metrics)
display(snapshot_metrics)



Validation metrics: {'roc_auc': 0.9932680538555692, 'pr_auc': 0.9927527128826358, 'brier': 0.03004127911790442, 'accuracy': 0.9876543209876543, 'precision': 0.9883720930232558, 'recall': 0.9883720930232558, 'f1': 0.9883720930232558, 'f2': 0.9883720930232558, 'predicted_positive_rate': 0.5308641975308642}
Test metrics: {'roc_auc': 0.99804821257362, 'pr_auc': 0.997593772184553, 'brier': 0.02387510471226536, 'accuracy': 0.9884057971014493, 'precision': 0.9865771812080537, 'recall': 0.9865771812080537, 'f1': 0.9865771812080537, 'f2': 0.9865771812080537, 'predicted_positive_rate': 0.4318840579710145}


,snapshot_date,users,target_rate,avg_score,score_std,predicted_positive_rate,split,target_rate_index
0,2026-01-08,76,0.539474,0.509953,0.424225,0.539474,val,1.000000
1,2026-01-15,86,0.523256,0.489586,0.418882,0.523256,val,0.969938
2,2026-01-22,98,0.489796,0.462901,0.419961,0.479592,test,1.000000
3,2026-01-29,112,0.437500,0.437189,0.417403,0.446429,test,0.893229
4,2026-02-05,135,0.385185,0.408698,0.408657,0.385185,test,0.786420


In [6]:
# Cell 5: Save model artifacts + drift baseline
model_version = datetime.now(UTC).strftime('v%Y%m%d%H%M%S')
model_name = 'sliceiq_reorder_propensity'
MODEL_DIR = ROOT / 'ml' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

artifact = {
    'model': best_model,
    'selected_features': selected_model,
    'selected_features_source': selected_features,
    'threshold': threshold,
    'threshold_policy': threshold_policy,
    'model_name': model_name,
    'model_version': model_version,
    'trained_at_utc': datetime.now(UTC).isoformat(),
}

model_path = MODEL_DIR / 'churn_reorder_model.joblib'
joblib.dump(artifact, model_path)

bins = np.linspace(0.0, 1.0, 11)
counts, _ = np.histogram(test_probs, bins=bins)
dist = (counts / max(1, counts.sum())).tolist()

drift_baseline = {
    'model_name': model_name,
    'model_version': model_version,
    'generated_at_utc': datetime.now(UTC).isoformat(),
    'bin_edges': bins.tolist(),
    'test_distribution': dist,
    'test_mean_score': float(np.mean(test_probs)),
    'test_std_score': float(np.std(test_probs)),
    'threshold': threshold,
    'threshold_policy': threshold_policy,
    'snapshot_score_baseline': snapshot_metrics[snapshot_metrics['split'] == 'test'][
        ['snapshot_date', 'avg_score', 'target_rate', 'predicted_positive_rate']
    ].to_dict(orient='records'),
}

baseline_path = MODEL_DIR / 'churn_drift_baseline.json'
baseline_path.write_text(json.dumps(drift_baseline, indent=2, default=str), encoding='utf-8')

metrics_payload = {
    'model_name': model_name,
    'model_version': model_version,
    'best_model_family': best_name,
    'selected_feature_count': len(selected_model),
    'selected_features': selected_model,
    'selected_features_source_count': len(selected_features),
    'threshold': threshold,
    'threshold_policy': threshold_policy,
    'candidate_validation_scores': val_table.to_dict(orient='records'),
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'dataset_rows': len(df),
    'train_rows': len(train_df),
    'val_rows': len(val_df),
    'test_rows': len(test_df),
    'class_balance': {
        'train_positive_rate': float(y_train.mean()),
        'val_positive_rate': float(y_val.mean()),
        'test_positive_rate': float(y_test.mean()),
    },
    'snapshot_split': {
        'train_start': train_df['snapshot_date'].min().isoformat(),
        'train_end': train_df['snapshot_date'].max().isoformat(),
        'val_start': val_df['snapshot_date'].min().isoformat(),
        'val_end': val_df['snapshot_date'].max().isoformat(),
        'test_start': test_df['snapshot_date'].min().isoformat(),
        'test_end': test_df['snapshot_date'].max().isoformat(),
    },
    'snapshot_metrics': snapshot_metrics.to_dict(orient='records'),
    'score_distribution_baseline': {
        'bin_edges': bins.tolist(),
        'test_distribution': dist,
        'test_mean_score': float(np.mean(test_probs)),
        'test_std_score': float(np.std(test_probs)),
    },
    'trained_at_utc': datetime.now(UTC).isoformat(),
}

metrics_path = MODEL_DIR / 'churn_reorder_metrics.json'
metrics_path.write_text(json.dumps(metrics_payload, indent=2, default=str), encoding='utf-8')

threshold_curve_path = MODEL_DIR / 'churn_threshold_curve.csv'
threshold_table.to_csv(threshold_curve_path, index=False)

print('Saved model:', model_path)
print('Saved metrics:', metrics_path)
print('Saved drift baseline:', baseline_path)
print('Saved threshold curve:', threshold_curve_path)



Saved model: /Users/deliorincon/Desktop/Sliceiq/ml/models/churn_reorder_model.joblib
Saved metrics: /Users/deliorincon/Desktop/Sliceiq/ml/models/churn_reorder_metrics.json
Saved drift baseline: /Users/deliorincon/Desktop/Sliceiq/ml/models/churn_drift_baseline.json
Saved threshold curve: /Users/deliorincon/Desktop/Sliceiq/ml/models/churn_threshold_curve.csv


In [7]:
# Cell 6: Feature importance + segment diagnostics
clf = best_model.named_steps['clf']
if hasattr(clf, 'coef_'):
    model_importance = np.abs(clf.coef_[0])
elif hasattr(clf, 'feature_importances_'):
    model_importance = clf.feature_importances_
else:
    model_importance = np.zeros(len(selected_model))

perm = permutation_importance(
    best_model,
    X_val,
    y_val,
    scoring='average_precision',
    n_repeats=12,
    random_state=42,
    n_jobs=1,
)

imp_df = pd.DataFrame(
    {
        'feature': selected_model,
        'model_importance': model_importance,
        'perm_importance': perm.importances_mean,
    }
)

ranking_cols = [c for c in ['feature', 'theme', 'final_score', 'perm_rank_pct'] if c in ranking_df.columns]
if ranking_cols:
    imp_df = imp_df.merge(ranking_df[ranking_cols], on='feature', how='left')

imp_df = imp_df.sort_values(['perm_importance', 'model_importance'], ascending=False).reset_index(drop=True)

imp_path = MODEL_DIR / 'churn_feature_importance.csv'
imp_df.to_csv(imp_path, index=False)

test_scored = test_df[['user_id', 'snapshot_date', 'will_order_next_30d']].copy()
test_scored['score'] = test_probs
test_scored['pred'] = (test_scored['score'] >= threshold).astype(int)

# Decile-level stability and lift checks.
test_scored['score_decile'] = pd.qcut(
    test_scored['score'].rank(method='first'),
    q=10,
    labels=False,
    duplicates='drop',
)
segment_diag = (
    test_scored.groupby('score_decile')
    .agg(
        users=('user_id', 'count'),
        churn_rate=('will_order_next_30d', 'mean'),
        avg_score=('score', 'mean'),
        pred_positive_rate=('pred', 'mean'),
    )
    .reset_index()
    .sort_values('score_decile', ascending=False)
)

segment_path = MODEL_DIR / 'churn_segment_diagnostics.csv'
segment_diag.to_csv(segment_path, index=False)

display(imp_df.head(30))
display(segment_diag)
print('Saved feature importance:', imp_path)
print('Saved segment diagnostics:', segment_path)




,feature,model_importance,perm_importance,theme,final_score,perm_rank_pct
0,log1p_std_order_value_lookback,0.155054,0.005699,value_mix,2.368334,0.844444
1,log1p_avg_gap_days_lookback,0.117948,0.003312,core,1.871110,0.888889
2,std_order_value_lookback_clip,0.104137,0.002547,value_mix,1.936448,0.733333
3,std_order_value_lookback,0.061895,0.001610,value_mix,1.917199,0.755556
4,max_gap_days_lookback,0.023376,0.001372,core,1.563782,0.977778
5,orders_30d_per_age,0.013630,0.001051,core,1.909221,0.955556
6,value_gap_ratio,0.028172,0.000944,core,1.849161,0.777778
7,avg_gap_days_lookback,0.039989,0.000931,core,1.317485,0.866667
8,orders_lookback,0.051642,0.000816,core,1.892659,0.622222
9,cohort_revenue_index,0.020870,0.000788,cohort,1.452184,0.444444


,score_decile,users,churn_rate,avg_score,pred_positive_rate
9,9,35,1.000000,0.982393,1.000000
8,8,34,1.000000,0.958073,1.000000
7,7,35,1.000000,0.913422,1.000000
6,6,34,0.970588,0.829155,1.000000
5,5,34,0.323529,0.342809,0.323529
4,4,35,0.028571,0.173829,0.000000
3,3,34,0.000000,0.087237,0.000000
2,2,35,0.000000,0.031138,0.000000
1,1,34,0.000000,0.010583,0.000000
0,0,35,0.000000,0.006546,0.000000


Saved feature importance: /Users/deliorincon/Desktop/Sliceiq/ml/models/churn_feature_importance.csv
Saved segment diagnostics: /Users/deliorincon/Desktop/Sliceiq/ml/models/churn_segment_diagnostics.csv


## Next Notebook

Proceed to `04_causal_inference_ab_did.ipynb`.
